# 03: Exploratory Data Analysis & Diagnostic Insights

**Project:** Digital Payment Failure Analysis  
**Objective:** Investigate failure rates across payment instruments, bank switches, time-of-day windows, and ticket amounts to uncover root-cause operational signatures.

In [ ]:
import pandas as pd
import numpy as np
from src.data_loader import load_processed_data
from src.feature_engineering import engineer_features
from src.metrics import calculate_overall_kpis, calculate_grouped_metrics, calculate_failure_reasons_breakdown
from src.analysis import run_full_exploratory_analysis

df = load_processed_data()
df = engineer_features(df)
kpis = calculate_overall_kpis(df)
for k, v in kpis.items():
    print(f'{k}: {v}')

### 1. Payment Method Breakdown
Compare failure rates across UPI, Credit Card, Debit Card, and NetBanking.

In [ ]:
pm_summary = calculate_grouped_metrics(df, 'payment_method')
pm_summary[['payment_method', 'total_transactions', 'failed_transactions', 'failure_rate_pct', 'avg_latency_ms']]

### 2. Bank Switch Performance
Rank core banking switches by reliability and failure rate.

In [ ]:
bank_summary = calculate_grouped_metrics(df, 'bank_switch')
bank_summary[['bank_switch', 'total_transactions', 'failed_transactions', 'failure_rate_pct', 'avg_latency_ms', 'p95_latency_ms']]

### 3. Cross-Tabulation Matrix: Bank Switch x Payment Method
Notice how SBI Cards (~38%) and HDFC UPI (~15%) suffer high failure rates.

In [ ]:
def matrix_fn(sub):
    tot = len(sub)
    fail = (sub['transaction_status'] == 'FAILED').sum()
    return round(fail / tot * 100.0, 2) if tot > 0 else 0.0

matrix = df.groupby(['bank_switch', 'payment_method']).apply(matrix_fn, include_groups=False).unstack()
matrix

### 4. Peak-Hour Switch Outage (HDFC 20:00 - 22:59)
Examine the severe failure rate spike and latency surge on HDFC switch.

In [ ]:
eda = run_full_exploratory_analysis(df)
print(eda['peak_hour_comparison'][['peak_failure_rate_pct', 'offpeak_failure_rate_pct', 'failure_rate_delta_pct', 'latency_delta_ms']])